In [12]:
import os
from dotenv import load_dotenv
load_dotenv()
MAP_KEY = os.getenv('MAP_KEY')
# Let's set your map key that was emailed to you. It should look something like 'abcdef1234567890abcdef1234567890'

# MAP_KEY = 'abcdef0123456789abcdef1234567890'

# now let's check how many transactions we have
import pandas as pd
import requests
url = 'https://firms.modaps.eosdis.nasa.gov/mapserver/mapkey_status/?MAP_KEY=' + MAP_KEY
try:
  response = requests.get(url)
  data = response.json()
  df = pd.Series(data)
  display(df)
except:
  # possible error, wrong MAP_KEY value, check for extra quotes, missing letters
  print ("There is an issue with the query. \nTry in your browser: %s" % url)



transaction_limit             5000
current_transactions            82
transaction_interval    10 minutes
dtype: object

In [14]:
# let's create a simple function that tells us how many transactions we have used.
# We will use this in later examples

def get_transaction_count() :
  count = 0
  try:
    response = requests.get(url)
    data = response.json()
    df = pd.Series(data)
    count = df['current_transactions']
  except:
    print ("Error in our call.")
  return count

tcount = get_transaction_count()
print ('Our current transaction count is %i' % tcount)

Our current transaction count is 77


In [15]:
# let's query data_availability to find out what date range is available for various datasets
# we will explain these datasets a bit later

# this url will return information about all supported sensors and their corresponding datasets
# instead of 'all' you can specify individual sensor, ex:LANDSAT_NRT
da_url = 'https://firms.modaps.eosdis.nasa.gov/api/data_availability/csv/' + MAP_KEY + '/all'
df = pd.read_csv(da_url)
display(df)

,data_id,min_date,max_date
0,MODIS_NRT,2025-10-01,2026-02-06
1,MODIS_SP,2000-11-01,2025-09-30
2,VIIRS_NOAA20_NRT,2025-11-01,2026-02-06
3,VIIRS_NOAA20_SP,2018-04-01,2025-10-31
4,VIIRS_NOAA21_NRT,2024-01-17,2026-02-06
5,VIIRS_SNPP_NRT,2025-11-01,2026-02-06
6,VIIRS_SNPP_SP,2012-01-20,2025-10-31
7,LANDSAT_NRT,2022-06-20,2026-02-06
8,GOES_NRT,2022-08-09,2026-02-06
9,BA_MODIS,2000-11-01,2025-10-01


In [ ]:
# in this example let's look at VIIRS NOAA-20, entire world and the most recent day
area_url = 'https://firms.modaps.eosdis.nasa.gov/api/area/csv/' + MAP_KEY + '/VIIRS_NOAA20_NRT/world/1'
start_count = get_transaction_count()
df_area = pd.read_csv(area_url)
end_count = get_transaction_count()
print ('We used %i transactions.' % (end_count-start_count))

df_area

We used 36 transactions.


,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight
0,8.99991,31.91796,367.00,0.41,0.61,2026-02-06,2,N20,VIIRS,h,2.0NRT,284.25,12.00,N
1,9.00060,31.91428,311.99,0.41,0.61,2026-02-06,2,N20,VIIRS,n,2.0NRT,274.92,3.23,N
2,9.04508,31.88822,303.85,0.41,0.60,2026-02-06,2,N20,VIIRS,n,2.0NRT,279.68,1.70,N
3,9.04519,31.88862,314.81,0.41,0.61,2026-02-06,2,N20,VIIRS,n,2.0NRT,276.70,2.63,N
4,9.05069,31.88966,304.09,0.41,0.61,2026-02-06,2,N20,VIIRS,n,2.0NRT,275.51,2.19,N
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32654,57.45420,-115.57198,326.26,0.50,0.40,2026-02-06,1950,N20,VIIRS,n,2.1URT,266.77,4.71,D
32655,57.45524,-115.54833,326.31,0.50,0.40,2026-02-06,1950,N20,VIIRS,n,2.1URT,267.00,4.56,D
32656,57.45567,-115.56490,342.68,0.50,0.40,2026-02-06,1950,N20,VIIRS,n,2.1URT,267.27,8.21,D
32657,57.45610,-115.58144,351.10,0.50,0.40,2026-02-06,1950,N20,VIIRS,n,2.1URT,268.27,8.92,D


In [ ]:
# Important variables:
# Coordinates - latitude and longitude
# frp - fire radiative power, higher = stronger fire
# confidence - low, nominal, high
# date and time
# day / night
df_area = df_area[['latitude', 'longitude', 'frp', 'confidence', 'acq_date', 'acq_time', 'daynight']]
df_area

,latitude,longitude,frp,confidence,acq_date,acq_time,daynight
0,8.99991,31.91796,12.00,h,2026-02-06,2,N
1,9.00060,31.91428,3.23,n,2026-02-06,2,N
2,9.04508,31.88822,1.70,n,2026-02-06,2,N
3,9.04519,31.88862,2.63,n,2026-02-06,2,N
4,9.05069,31.88966,2.19,n,2026-02-06,2,N
...,...,...,...,...,...,...,...
32654,57.45420,-115.57198,4.71,n,2026-02-06,1950,D
32655,57.45524,-115.54833,4.56,n,2026-02-06,1950,D
32656,57.45567,-115.56490,8.21,n,2026-02-06,1950,D
32657,57.45610,-115.58144,8.92,n,2026-02-06,1950,D
